# Enhanced RAG System Demo
## Task 1: Traceability + Task 2: Conceptual Reasoning

This notebook demonstrates:
1. **Option A**: Check pre-ingested data (fast)
2. **Option B**: Run ingestion live (slow but comprehensive)
3. **Query System**: Ask questions and see source citations

---


## Setup: Import Required Libraries


In [8]:
import os
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import OpenAI

# Configuration - use absolute path to vector store
DB_CHROMA_PATH = "/Users/ashutoshkumv/Documents/gAi/vector_stores/db_chroma_code"
EMBEDDINGS_MODEL = "thenlper/gte-large"

print("✅ Libraries imported successfully!")
print(f"📁 Vector Store Path: {DB_CHROMA_PATH}")


✅ Libraries imported successfully!
📁 Vector Store Path: /Users/ashutoshkumv/Documents/gAi/vector_stores/db_chroma_code


---
## Part 1: Check Pre-Ingested Data
### ⚡ Quick option for demos - shows existing vector store


In [9]:
def check_ingested_data():
    """
    Check if we have pre-ingested data and show statistics.
    This demonstrates that ingestion has already been done.
    """
    print("="*70)
    print("CHECKING PRE-INGESTED DATA")
    print("="*70)
    
    # Debug: show the path being checked
    print(f"\n🔍 Looking for data at:")
    print(f"   {DB_CHROMA_PATH}")
    
    if not os.path.exists(DB_CHROMA_PATH):
        print(f"\n❌ No data found at: {DB_CHROMA_PATH}")
        print("   Run Part 2 (ingestion) first!")
        return None
    
    try:
        # Load vector store
        embeddings = HuggingFaceEmbeddings(
            model_name=EMBEDDINGS_MODEL,
            model_kwargs={'device': 'cpu'}
        )
        vectordb = Chroma(
            persist_directory=DB_CHROMA_PATH, 
            embedding_function=embeddings
        )
        
        # Get stats
        collection = vectordb._collection
        count = collection.count()
        
        print(f"\n✅ Pre-ingested data found!\n")
        print(f"📊 Vector Store Statistics:")
        print(f"   • Total chunks: {count}")
        
        # Sample documents to check metadata
        if count > 0:
            sample_docs = vectordb.similarity_search("import", k=5)
            
            source_files = set()
            has_summaries = 0
            
            for doc in sample_docs:
                source = doc.metadata.get('source', 'unknown')
                source_files.add(source)
                if 'file_summary' in doc.metadata:
                    has_summaries += 1
            
            print(f"   • Source files (in sample): {len(source_files)}")
            print(f"   • Chunks with summaries: {has_summaries}/{len(sample_docs)}")
            
            print(f"\n📁 Sample Source Files:")
            for src in list(source_files)[:3]:
                print(f"   • {os.path.basename(src)}")
            
            # Show metadata structure
            print(f"\n🔍 Sample Chunk Metadata:")
            doc = sample_docs[0]
            print(f"   • Source: {os.path.basename(doc.metadata.get('source', 'unknown'))}")
            print(f"   • Content preview: {doc.page_content[:100]}...")
            
            if 'file_summary' in doc.metadata:
                print(f"   • File summary: {doc.metadata['file_summary'][:150]}...")
                print(f"   ✅ Task 2 metadata present!")
            else:
                print(f"   ⚠️ No file summary found")
        
        return vectordb
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return None

# Run the check
vectordb = check_ingested_data()


CHECKING PRE-INGESTED DATA

🔍 Looking for data at:
   /Users/ashutoshkumv/Documents/gAi/vector_stores/db_chroma_code

✅ Pre-ingested data found!

📊 Vector Store Statistics:
   • Total chunks: 311
   • Source files (in sample): 5
   • Chunks with summaries: 0/5

📁 Sample Source Files:
   • tok_eval.py
   • arc.py
   • customjson.py

🔍 Sample Chunk Metadata:
   • Source: humaneval.py
   • Content preview: def extract_imports(prompt):
    """Extract import statements from the beginning of a code block."""...
   ⚠️ No file summary found
